# Auto-detecting distributions and ML models

You asked the right question: **don't libraries already exist that do this for you?**

**Yes. They do.** This notebook shows you the two main ones in action on a real dataset.

- **`fitter`** — finds the best-fitting distribution for a column of data
- **`lazypredict`** — finds the best-performing ML model for a prediction task

We use the `tips` dataset (restaurant tips data — 244 rows, available in seaborn).


## Setup

Install the two libraries we need (just once):

```
pip install fitter lazypredict
```

The rest (pandas, numpy, scipy, seaborn, plotly, scikit-learn) should already be installed from the previous notebook.


In [10]:
# Standard imports
import numpy as np
import pandas as pd
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats

import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("Setup done.")


Setup done.


## Load and look at the data

`tips` records restaurant bills and tips, with some context (day, time, party size, etc.). Our two questions:
1. What's the distribution of `total_bill`?
2. Can we predict `tip` from the other columns? Which model works best?


In [11]:
tips = sns.load_dataset('tips')
print(tips.shape)
tips.head()


(244, 7)


,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


In [12]:
# Quick look at total_bill
fig = px.histogram(tips, x='total_bill', nbins=40,
                   title='Distribution of total_bill')
fig.show()


**Eyeballing it**: looks right-skewed, positive only — your gut should say "this looks log-normal-ish." Let's let `fitter` confirm or correct you.


---
## Part 1 — Auto-detect the best distribution with `fitter`

`fitter` tries dozens of distributions, scores each by how well it fits (using a "sum of squared errors" — lower = better), and ranks them.

By default it tests 80+ distributions, which is slow. We'll restrict to common ones to keep it quick.


In [13]:
from fitter import Fitter

# Candidates we care about (all from scipy.stats)
candidates = ['norm', 'lognorm', 'expon', 'gamma', 'beta', 'weibull_min', 'chi2']

f = Fitter(tips['total_bill'].values, distributions=candidates, timeout=30)
f.fit()

# Top 5 best-fitting distributions, ranked
print("=== Best-fitting distributions (lower sumsquare_error = better) ===")
print(f.summary(Nbest=5))


2026-06-11 19:15:12.287 | INFO     | fitter.fitter:_fit_single_distribution:408 - Fitted norm: error=0.028072, AIC=1762.37, KS=0.1188
2026-06-11 19:15:12.336 | INFO     | fitter.fitter:_fit_single_distribution:408 - Fitted expon: error=0.057846, AIC=1866.39, KS=0.2725


BrokenProcessPool: A result has failed to un-serialize. Please ensure that the objects returned by the function are always picklable.

**Reading the table**

| Column | What it means |
|---|---|
| `sumsquare_error` | How far the fitted curve is from the actual histogram. **Lower is better.** |
| `aic`, `bic` | Information criteria — also lower is better. They penalize complexity. |
| `kl_div` | Another distance measure |

Whichever distribution sits at the top is `fitter`'s pick. If your gut said "log-normal" and `lognorm` ranks #1 — you've correctly heard the music.


In [ ]:
# What did fitter actually find?
best_dist_name = list(f.get_best().keys())[0]
best_params = f.get_best()[best_dist_name]
print(f"Best distribution: {best_dist_name}")
print(f"Fitted parameters: {best_params}")


In [ ]:
# Plot the empirical histogram with the fitted curve overlaid
from scipy.stats import lognorm

x_range = np.linspace(tips['total_bill'].min(), tips['total_bill'].max(), 200)

if best_dist_name == 'lognorm':
    pdf = lognorm.pdf(x_range, **best_params)
else:
    dist = getattr(stats, best_dist_name)
    pdf = dist.pdf(x_range, **best_params)

fig = go.Figure()
fig.add_histogram(x=tips['total_bill'], histnorm='probability density',
                  nbinsx=40, name='Data', marker_color='lightsteelblue')
fig.add_scatter(x=x_range, y=pdf, mode='lines',
                name=f'Fitted {best_dist_name}',
                line=dict(color='crimson', width=2.5))
fig.update_layout(title=f'total_bill — empirical vs fitted {best_dist_name}',
                  xaxis_title='total_bill', yaxis_title='Density', height=400)
fig.show()


**That's it for distribution detection.** Three lines of `fitter` and you have a ranked answer.

**What you'd do with this info** in a real pipeline:
- `total_bill` is log-normal → log-transform it before using it as a feature: `np.log(total_bill)`
- This makes downstream linear models / neural nets work better
- It also gives more honest prediction intervals


---
## Part 2 — Auto-select the best ML model with `lazypredict`

Now the real question: we want to **predict `tip`** from the other columns. Which ML model should we use?

`lazypredict` tries 25+ regression models and ranks them by performance. We just hand it the data.


In [ ]:
# Prepare data — one-hot encode the categorical columns, split into train/test
from sklearn.model_selection import train_test_split

# Predict 'tip' using everything else
X = tips.drop(columns=['tip'])
y = tips['tip']

# One-hot encode the categoricals (sex, smoker, day, time)
X = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

print(f"Training set: {X_train.shape}  ·  Test set: {X_test.shape}")
print(f"Features: {list(X.columns)}")


In [ ]:
# Run lazypredict — this is the magic
from lazypredict.Supervised import LazyRegressor

reg = LazyRegressor(verbose=0, ignore_warnings=True)
models_df, predictions = reg.fit(X_train, X_test, y_train, y_test)

print("=== Ranked ML models (best R² at top) ===")
print(models_df.head(10))


**Reading the table**

| Column | What it means |
|---|---|
| `R-Squared` | How much of the variance the model explains. **1.0 = perfect, 0 = useless.** |
| `Adjusted R-Squared` | R² penalized for using lots of features |
| `RMSE` | Average prediction error (in the units of the target — here, dollars) |
| `Time Taken` | How long the model took to fit |

The model at the top is `lazypredict`'s pick. Often it's a tree-based ensemble (Gradient Boosting, Random Forest, XGBoost). Sometimes a simple Linear Regression wins — and that's actually a useful signal that your data is well-behaved.


In [ ]:
# Visualize the top 10 models
top10 = models_df.head(10).reset_index()
fig = px.bar(top10, x='R-Squared', y='Model', orientation='h',
             title='Top 10 models by R²',
             color='R-Squared', color_continuous_scale='Viridis')
fig.update_layout(yaxis=dict(autorange='reversed'), height=500)
fig.show()


---
## Bonus — the DIY version (no library needed)

Here's what `lazypredict` is doing *under the hood*. Knowing this is what makes you good — once you see it, the magic disappears.

It's just a loop.


In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import cross_val_score

# A dictionary of candidate models
candidates = {
    'Linear Regression':   LinearRegression(),
    'Ridge':               Ridge(),
    'Lasso':               Lasso(),
    'Decision Tree':       DecisionTreeRegressor(random_state=42),
    'Random Forest':       RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting':   GradientBoostingRegressor(random_state=42),
    'SVM':                 SVR(),
    'k-NN':                KNeighborsRegressor(),
}

# Try each, record cross-validated R²
results = []
for name, model in candidates.items():
    scores = cross_val_score(model, X, y, cv=5, scoring='r2')
    results.append({'Model': name,
                    'Mean R²': scores.mean(),
                    'Std R²': scores.std()})

results_df = pd.DataFrame(results).sort_values('Mean R²', ascending=False)
print(results_df.to_string(index=False))


**That's the whole AutoML idea.** It's a loop over models, each scored by cross-validation. `lazypredict` and `pycaret` are just nicer wrappers around this pattern with more models pre-configured.


---
## The honest takeaway

Yes — **the functions you suspected exist actually do exist.** You don't have to compute fits or compare models by hand. The libraries handle the grinding.

**So what's the point of understanding the distributions then?**

Three things AutoML tools cannot do for you:

**1. They can't tell you when the answer is wrong.**
`fitter` might rank `chi2` slightly above `lognorm` on a tiny dataset. Without knowing the *story* behind log-normal (multiplicative effects → income, prices, file sizes), you'd just believe the table. Domain knowledge is the tiebreaker.

**2. They can't pick the right metric.**
`lazypredict` ranks by R². But if your data is right-skewed and you care about big values, R² can be misleading. If your classes are imbalanced, accuracy is misleading. *You* have to know which metric matches what you actually want.

**3. They can't do the data engineering.**
The model selection only finds the best of what's possible *given your features*. If you didn't log-transform a log-normal feature, the "best" model is still suboptimal. The biggest wins in real ML come from feature engineering decisions guided by understanding the data — which is the Part-1-of-this-notebook work.

**Practical workflow that actually works:**

1. Look at each column's distribution (use `fitter` for the math)
2. Decide on transformations based on what you saw (log-transform skewed positives, etc.)
3. Run `lazypredict` to see which model family is winning
4. Tune the winner properly (with `GridSearchCV` or `Optuna`)
5. Sanity-check predictions against domain knowledge

The libraries do steps 1 and 3 in one line each. **Steps 2, 4, and 5 are why your job exists.**
